# 04 - Ingestão Bronze: Micro-Lotes em Delta Lake com Metadados
**Squad 2 — Real Time for Business | Dupla 1**  
**Integrantes:** Lucas Sousa Santos Oliveira & Zaiden Emiliano Segundo Seleme  
**Tabelas de Escopo:** `ecommerce_produtos` e `ecommerce_categorias`  
**Branch:** `feat/squad2-lucas_zaiden`  

### Objetivo da Task (Sprint 2 - Task 1):
1. **Detecção Incremental via Auto Loader:** Identificar novos arquivos `.parquet` depositados no bucket `raw/real-time-data/` utilizando Spark Structured Streaming com Auto Loader (`format("cloudFiles")`) e disparo sob demanda `.trigger(availableNow=True)`.
2. **Auditoria e Rastreabilidade:** Adicionar em cada registro as colunas obrigatórias:
   * `bronze_ingested_at`: timestamp do momento da ingestão via `current_timestamp()`.
   * `bronze_source_file`: nome/caminho completo do arquivo de origem via `input_file_name()`.
3. **Zero Transformação (Regra de Ouro Bronze):** Não aplicar limpeza, filtros de regras de negócio ou alteração de tipos. O dado bruto é preservado 100% íntegro.
4. **Particionamento Temporal Obrigatório:** Salvar em formato **Delta Lake** em modo **`append`**, particionado por data de ingestão (`ano`, `mes`, `dia`, `hora`).
5. **Isolamento de Namespace (`/grupo1/`):** Gravar os dados no container `squad2` sob o prefixo seguro `/grupo1/` para evitar conflito com outras duplas.
6. **Controle de Idempotência e Metadados:** Registrar os lotes e arquivos processados na tabela Delta de controle `squad2.ingestion_control_log`.

## 1. Carregamento Seguro das Credenciais e Variáveis de Ambiente

In [0]:
import os
from dotenv import load_dotenv, find_dotenv

# Carregamento automático do arquivo .env com override
dotenv_path = find_dotenv()
if not dotenv_path:
    candidatos = [
        os.path.join(os.getcwd(), ".env"),
        os.path.join(os.path.dirname(os.getcwd()), ".env"),
        os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), ".env")
    ]
    for c in candidatos:
        if os.path.exists(c):
            dotenv_path = c
            break

load_dotenv(dotenv_path, override=True)

storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME", "internshipdatalake")
client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")

print("Ambiente configurado:")
print(f"  Storage Account: {storage_account}")
print(f"  Client ID disponível: {client_id is not None}")
print(f"  Tenant ID disponível: {tenant_id is not None}")
print(f"  Client Secret disponível: {client_secret is not None}")

## 2. Configurações de Conexão OAuth e Definição dos Caminhos ABFSS
No Databricks Serverless / Spark Connect, credenciais Hadoop (como `fs.azure...`) NÃO podem ser injetadas globalmente via `spark.conf.set`. Elas são passadas diretamente via `.options(**adls_options)` em cada operação distribuída.

In [0]:
# Configurações OAuth do Service Principal para injeção granular nas operações do cluster
adls_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

# Definição dos caminhos com isolamento de namespace no prefixo /grupo1/
base_raw = f"abfss://raw@{storage_account}.dfs.core.windows.net/real-time-data"
base_squad = f"abfss://squad2@{storage_account}.dfs.core.windows.net/grupo1"

caminhos = {
    "origem_produtos": f"{base_raw}/*/*/*/*/ecommerce_produtos.parquet",
    "origem_categorias": f"{base_raw}/*/*/*/*/ecommerce_categorias.parquet",
    "bronze_produtos": f"{base_squad}/bronze/ecommerce_produtos",
    "bronze_categorias": f"{base_squad}/bronze/ecommerce_categorias",
    "checkpoint_bronze_produtos": f"{base_squad}/checkpoints/bronze_produtos",
    "checkpoint_bronze_categorias": f"{base_squad}/checkpoints/bronze_categorias",
    "schema_produtos": f"{base_squad}/checkpoints/schema_bronze_produtos",
    "schema_categorias": f"{base_squad}/checkpoints/schema_bronze_categorias",
    "metadata_control_log": f"{base_squad}/metadata/ingestion_control_log"
}

print("Caminhos configurados no Data Lake:")
for k, v in caminhos.items():
    print(f"  {k}: {v}")
print("\nConfigurações adls_options prontas para injeção granular.")

## 3. Schemas Estruturais de Entrada e Função de Log de Controle
Definir schemas explícitos acelera o arranque do Auto Loader e garante validação estrutural imediata.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, BooleanType, LongType, TimestampType
from pyspark.sql.functions import current_timestamp, input_file_name, year, month, dayofmonth, hour, col, lit

# Schema para ecommerce_produtos
schema_produtos = StructType([
    StructField("sku", StringType(), True),
    StructField("nome_produto", StringType(), True),
    StructField("descricao", StringType(), True),
    StructField("id_categoria", StringType(), True),
    StructField("preco_lista", DoubleType(), True),
    StructField("unidade_medida", StringType(), True),
    StructField("nome_marca", StringType(), True),
    StructField("is_ativo", BooleanType(), True)
])

# Schema para ecommerce_categorias
schema_categorias = StructType([
    StructField("id_categoria", StringType(), True),
    StructField("nome_categoria", StringType(), True),
    StructField("id_categoria_pai", StringType(), True),
    StructField("nome_categoria_pai", StringType(), True),
    StructField("tipo_categoria", StringType(), True)
])

# Função hook para registrar micro-lotes processados na tabela Delta de controle
def registrar_log_microbatch(df_batch, batch_id, tabela_origem, caminho_delta_destino):
    total_linhas = df_batch.count()
    if total_linhas == 0:
        return

    # Coletar arquivos únicos processados neste micro-lote
    arquivos = df_batch.select("bronze_source_file").distinct().collect()
    lista_arquivos = [r["bronze_source_file"] for r in arquivos]

    print(f"\n>>> [Micro-Lote {batch_id}] Ingerindo {total_linhas} linhas na Bronze de '{tabela_origem}'...")
    for arq in lista_arquivos:
        print(f"    Arquivo de origem: {arq}")

    # 1. Escrever o lote na tabela Bronze particionada com autenticação granular
    (df_batch.write
        .format("delta")
        .options(**adls_options)
        .mode("append")
        .partitionBy("ano", "mes", "dia", "hora")
        .save(caminho_delta_destino))

    # 2. Criar e anexar o registro de auditoria na tabela de controle com autenticação granular
    log_rows = []
    for arq in lista_arquivos:
        log_rows.append((
            tabela_origem,
            arq,
            int(batch_id),
            int(total_linhas),
            "SUCCESS"
        ))

    schema_log = StructType([
        StructField("tabela_origem", StringType(), False),
        StructField("arquivo_processado", StringType(), False),
        StructField("batch_id", LongType(), False),
        StructField("total_linhas_lote", LongType(), False),
        StructField("status", StringType(), False)
    ])

    df_log = (spark.createDataFrame(log_rows, schema=schema_log)
              .withColumn("timestamp_processamento", current_timestamp()))

    (df_log.write
        .format("delta")
        .options(**adls_options)
        .mode("append")
        .save(caminhos["metadata_control_log"]))
    
    print(f"<<< [Micro-Lote {batch_id}] Concluído e auditado com sucesso na tabela de controle.")

print("Função de auditoria de micro-lote e schemas definidos com sucesso.")

## 4. Ingestão Incremental Bronze: `ecommerce_produtos`
Lê os micro-lotes do bucket `raw`, adiciona colunas de auditoria (`bronze_ingested_at`, `bronze_source_file`), deriva partições temporais e grava em Delta Bronze via `append`.

In [0]:
print("Iniciando Auto Loader para 'ecommerce_produtos'...")

# Configurar leitura streaming com Auto Loader e adls_options
stream_produtos_raw = (spark.readStream
    .format("cloudFiles")
    .options(**adls_options)
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", caminhos["schema_produtos"])
    .option("pathGlobFilter", "*ecommerce_produtos.parquet")
    .schema(schema_produtos)
    .load(base_raw))

# Enriquecer com metadados de auditoria e colunas derivadas para particionamento temporal
df_produtos_bronze = (stream_produtos_raw
    .withColumn("bronze_ingested_at", current_timestamp())
    .withColumn("bronze_source_file", input_file_name())
    .withColumn("ano", year(col("bronze_ingested_at")))
    .withColumn("mes", month(col("bronze_ingested_at")))
    .withColumn("dia", dayofmonth(col("bronze_ingested_at")))
    .withColumn("hora", hour(col("bronze_ingested_at"))))

# Executar escrita com checkpoint e trigger availableNow
query_produtos = (df_produtos_bronze.writeStream
    .format("delta")
    .options(**adls_options)
    .outputMode("append")
    .option("checkpointLocation", caminhos["checkpoint_bronze_produtos"])
    .trigger(availableNow=True)
    .foreachBatch(lambda df, b_id: registrar_log_microbatch(
        df, b_id, "ecommerce_produtos", caminhos["bronze_produtos"]
    ))
    .start())

query_produtos.awaitTermination()
print("Ingestão Bronze de 'ecommerce_produtos' concluída com sucesso!")

## 5. Ingestão Incremental Bronze: `ecommerce_categorias`
Executa o mesmo processo de ingestão atômica e auditoria para a tabela de categorias.

In [0]:
print("Iniciando Auto Loader para 'ecommerce_categorias'...")

# Configurar leitura streaming com Auto Loader e adls_options
stream_categorias_raw = (spark.readStream
    .format("cloudFiles")
    .options(**adls_options)
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", caminhos["schema_categorias"])
    .option("pathGlobFilter", "*ecommerce_categorias.parquet")
    .schema(schema_categorias)
    .load(base_raw))

# Enriquecer com metadados de auditoria e colunas derivadas para particionamento temporal
df_categorias_bronze = (stream_categorias_raw
    .withColumn("bronze_ingested_at", current_timestamp())
    .withColumn("bronze_source_file", input_file_name())
    .withColumn("ano", year(col("bronze_ingested_at")))
    .withColumn("mes", month(col("bronze_ingested_at")))
    .withColumn("dia", dayofmonth(col("bronze_ingested_at")))
    .withColumn("hora", hour(col("bronze_ingested_at"))))

# Executar escrita com checkpoint e trigger availableNow
query_categorias = (df_categorias_bronze.writeStream
    .format("delta")
    .options(**adls_options)
    .outputMode("append")
    .option("checkpointLocation", caminhos["checkpoint_bronze_categorias"])
    .trigger(availableNow=True)
    .foreachBatch(lambda df, b_id: registrar_log_microbatch(
        df, b_id, "ecommerce_categorias", caminhos["bronze_categorias"]
    ))
    .start())

query_categorias.awaitTermination()
print("Ingestão Bronze de 'ecommerce_categorias' concluída com sucesso!")

## 6. Criação e Mapeamento das Tabelas Externas no Databricks Metastore
Registra as tabelas externas sob o schema `squad2` apontando diretamente para os diretórios `/grupo1/` no Data Lake.

In [0]:
try:
    # Garantir existência do schema
    spark.sql("CREATE SCHEMA IF NOT EXISTS squad2")

    # Registrar tabela externa de produtos
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS squad2.bronze_ecommerce_produtos
    USING DELTA
    LOCATION '{caminhos["bronze_produtos"]}'
    """)

    # Registrar tabela externa de categorias
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS squad2.bronze_ecommerce_categorias
    USING DELTA
    LOCATION '{caminhos["bronze_categorias"]}'
    """)

    # Registrar tabela externa de controle de ingestão
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS squad2.ingestion_control_log
    USING DELTA
    LOCATION '{caminhos["metadata_control_log"]}'
    """)
    print("Tabelas externas registradas com sucesso no schema squad2!")
except Exception as e:
    print(f"Nota sobre Metastore Externo: {e}")
    print("Os dados Delta no ADLS e views temporárias estão prontos para consulta local.")

## 7. Auditoria Pós-Carga e Validação das Partições Físicas

In [0]:
print("=== Auditoria da Camada Bronze ===")

# 1. Leitura direta via Delta no Data Lake com autenticação granular adls_options
df_check_prod = spark.read.format("delta").options(**adls_options).load(caminhos["bronze_produtos"])
df_check_cat = spark.read.format("delta").options(**adls_options).load(caminhos["bronze_categorias"])
df_check_log = spark.read.format("delta").options(**adls_options).load(caminhos["metadata_control_log"])

# Criar views temporárias para consultas SQL diretas
df_check_prod.createOrReplaceTempView("bronze_ecommerce_produtos")
df_check_cat.createOrReplaceTempView("bronze_ecommerce_categorias")
df_check_log.createOrReplaceTempView("ingestion_control_log")

count_prod = df_check_prod.count()
count_cat = df_check_cat.count()
count_log = df_check_log.count()

print(f"Total de registros em bronze_ecommerce_produtos:   {count_prod}")
print(f"Total de registros em bronze_ecommerce_categorias: {count_cat}")
print(f"Total de registros na tabela de controle de logs:   {count_log}")

# 2. Verificação das partições temporais criadas
print("\n--- Partições Temporais Detectadas em Produtos ---")
display(df_check_prod.groupBy("ano", "mes", "dia", "hora").count().orderBy("ano", "mes", "dia", "hora"))

# 3. Exibir amostra dos metadados de auditoria
print("\n--- Amostra de Registros da Camada Bronze (Produtos) ---")
display(df_check_prod.select("sku", "nome_produto", "preco_lista", "bronze_ingested_at", "bronze_source_file").limit(5))

# 4. Exibir o histórico da tabela de controle de arquivos
print("\n--- Histórico de Ingestão (ingestion_control_log) ---")
display(df_check_log.orderBy(col("timestamp_processamento").desc()))